# MoodNote-AI — Sinh dữ liệu nhật ký giả lập (Phase 3, Colab)

**CHƯA chạy/xác nhận trong phiên code này** — chủ nhiệm đề tài tự chạy trên Colab
thật với GPU T4 (Runtime > Change runtime type > T4 GPU) + HF token của mình.

Notebook này sinh dữ liệu hàng loạt (~2.800 mẫu) TUẦN TỰ cho 2 model:
Llama-3-8B-Instruct rồi mới đến Qwen3-8B (đúng hạ tầng đã chốt — T4 16GB không đủ để
giữ cả 2 model 4-bit cùng lúc trong bộ nhớ).

## 1. Lấy code của repo lên Colab

Điền URL remote git của bạn (hoặc mount Google Drive rồi copy thư mục `src/`,
`configs/` vào `/content/MoodNote-AI`). Sau bước này, thư mục làm việc phải có
`src/data/synthetic/`, `src/qa/`, `configs/datagen_config.yaml`.

In [ ]:
%cd /content
!git clone https://github.com/ToanHuynh0201/MoodNote-AI.git MoodNote-AI
%cd MoodNote-AI
!git checkout feature/ToanHuynh/EnhanceForNCKH

REPO_ROOT = "/content/MoodNote-AI"  # sửa nếu bạn đặt ở nơi khác

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas pyyaml pydantic huggingface_hub

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

# Llama-3-8B-Instruct là gated model trên HuggingFace — cần token đã được cấp quyền.
os.environ["HF_TOKEN"] = getpass("Nhập HF_TOKEN: ")
login(token=os.environ["HF_TOKEN"])

In [ ]:
import sys

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.utils.config import load_config

datagen_config = load_config("configs/datagen_config.yaml")
datagen_config["models"]

## 2. Sinh dữ liệu — Llama-3-8B-Instruct trước

In [ ]:
from src.data.synthetic.generate import generate_dataset
from src.data.synthetic.llm_client import HFLocalClient

llama_cfg = datagen_config["models"]["llama"]
llama_client = HFLocalClient(model_id=llama_cfg["bulk_model_id"], load_in_4bit=True)

generate_dataset(
    client=llama_client,
    model_display_name=llama_cfg["display_name"],
    channel="hf_colab",
    output_path="data/synthetic/raw/llama3_round1.jsonl",
    generation_round=1,
)

In [ ]:
# Giải phóng Llama khỏi bộ nhớ GPU trước khi tải Qwen — T4 16GB không đủ giữ cả 2.
import gc

import torch

del llama_client
gc.collect()
torch.cuda.empty_cache()

## 3. Sinh dữ liệu — Qwen3-8B

In [ ]:
qwen_cfg = datagen_config["models"]["qwen"]
qwen_client = HFLocalClient(model_id=qwen_cfg["bulk_model_id"], load_in_4bit=True)

generate_dataset(
    client=qwen_client,
    model_display_name=qwen_cfg["display_name"],
    channel="hf_colab",
    output_path="data/synthetic/raw/qwen3_round1.jsonl",
    generation_round=1,
)

## 4. Lưu kết quả về Google Drive

Colab free tier không giữ ổ đĩa giữa các phiên — tải `data/synthetic/raw/*.jsonl` về
máy hoặc lưu vào Drive trước khi phiên Colab kết thúc.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/MoodNote-AI/data/synthetic/raw"
!cp data/synthetic/raw/*.jsonl "/content/drive/MyDrive/MoodNote-AI/data/synthetic/raw/"

## 5. Bước tiếp theo (chạy ở máy local, KHÔNG chạy trên Colab)

1. Tải `*.jsonl` ở trên về `data/synthetic/raw/` trong repo local.
2. `python scripts/dedup_and_filter.py --input data/synthetic/raw/llama3_round1.jsonl data/synthetic/raw/qwen3_round1.jsonl`
3. `python scripts/check_leakage.py --input data/synthetic/dedup/deduped.jsonl`
4. `python scripts/export_audit_sample.py --input data/synthetic/leakage_checked/clean.jsonl` — gửi 2 sheet cho tác giả + cộng tác viên điền tay.
5. `python scripts/compute_agreement.py --rater-a ... --rater-b ...`
6. `python scripts/cross_llm_audit.py --input data/synthetic/leakage_checked/clean.jsonl --llama-channel openrouter --qwen-channel hf_colab` (hoặc chạy phần cross-LLM ngay trên Colab này nếu tiện).
7. `python scripts/apply_acceptance_gate.py --clean-samples ... --cross-llm-reviews ... --kappa-report ...`